# Creating the Binary Ground Truth Cold Wave Days
Version 14 December 2022, Selina Kiefer

### Input: netcdf- or grib-file and csv-file
1. netcdf- or grib-file with ground truth (i.e. E-OBS V23.1e, tg, daily mean, 1950 - 2020, Nov-Apr, 3-20°E and 45-60°N, e.g. from https://www.ecad.eu/download/ensembles/download.php ), or, optionally, directly continuous timeseries of ground truth temperature in csv-format (then skip the part with CDO and start directly with Python)
2. threshold temperatures for cold waves in csv-format
### Output: csv-file
binary timeseries of cold wave days in csv-format (1 = cold wave day, 0 = non cold wave day)

## Used software: Climate Data Operators and Python

#### Climate Data Operators (CDO) 

Taylored open-source software to perform the most-common meteorological operations efficiently (and much faster than Python). 

Up to date information about CDO: https://code.mpimet.mpg.de/projects/cdo

Reference: Schulzweida, U. (2019): "CDO User Guide". Available at: https://doi.org/10.5281/ZENODO.3539275.

#### Short introduction to CDO

The overall structure for most operations is:

cdo -operator_last_executed,optional_specifications -operator_first_executed,optional_specifcations ifile ofile

e.g. cdo -daymean -selyear,1950,1951 input_file_name output_file_name

The input file (ifile) and the output file (ofile) of one operation have to have different names. So it is best to name all files, which are not intended for further use, similarly, e.g. temp_1, temp_2, etc. and to delete them afterwards directly.

CDO does not ask when overwriting an existing file. So make sure that everything is named uniquely and correctly.

### Start with CDO

Since it is much faster than Python.

#### At first, check the data file's content
This is optional.

In [1]:
# Short overview of the data file's content.
!cdo sinfov ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.nc

   File format : NetCDF4
    -1 : Institut Source   T Steptype Levels Num    Points Num Dtype : Parameter name
     1 : unknown  unknown  v instant       1   1     25500   1  I16  : tg            
   Grid coordinates :
     1 : lonlat                   : points=25500 (170x150)
                        longitude : 3.04986 to 19.94986 by 0.1 degrees_east
                         latitude : 45.04986 to 59.94986 by 0.1 degrees_north
   Vertical coordinates :
     1 : surface                  : levels=1
   Time coordinate :  25933 steps
     RefTime =  1950-01-01 00:00:00  Units = days  Calendar = standard  Bounds = true
  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss
  1950-01-01 00:00:00  1950-01-02 00:00:00  1950-01-03 00:00:00  1950-01-04 00:00:00
  1950-01-05 00:00:00  1950-01-06 00:00:00  1950-01-07 00:00:00  1950-01-08 00:00:00
  1950-01-09 00:00:00  1950-01-10 00:00:00  1950-01-11 00:00:00  1950-01-12 00:00:00
  1950-01-13 00:00:00  1950-01-14 00:

In [2]:
# Optional: use a more detailed description of the data file's content. It might be wise to use 
# a separate terminal for this command since it prints all available information about the data
# file. Use grib_dump for files in grib-format, 
# nc_dump for files in netcdf-format. 
#! grib_dump ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.grib
#! nc_dump ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.nc

#### Spatial Preprocessing 

In [3]:
# Selection of a gridbox (sellonlatbox,°W,°E,°S,°N). Western longitudes have to be given as 
# 360°-°W). In case there is only 1 latitude or longitude to average over, select the desired
# longitude/latitude and on the second position the desired longitude/latitude+1. Otherwise 
# CDO may perform not well.    
! cdo sellonlatbox,3,20,45,60 ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.nc temp_1

cdo    sellonlatbox: Processed 661291500 values from 1 variable over 25933 timesteps [15.87s 121MB].


In [4]:
# Calculation of the areal mean (fldmean) over the desired area chosen above.
! cdo fldmean temp_1 temp_2

cdo    fldmean:                        1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 7 7 7 7 7 7 7 7 7 7 8 8 8 8 8 8 8 8 8 8 9 9 9 9 9 9 9 9 9 910cdo    fldmean: Processed 661291500 values from 1 variable over 25933 timesteps [8.53s 106MB].


#### Temporal Preprocessing

In [5]:
# Selection of certain times, e.g. only the wintermonths (selmon).
! cdo selmon,1,2,3,4,11,12 temp_2 temp_3

cdo    selmonth: Processed 12869 values from 1 variable over 25933 timesteps [1.97s 81MB].


In [6]:
# Remove the lead time from the beginning of the data. 
# Number of days to delete = lead_time.
! cdo delete,day=1,2,3,4,5,6,7,8,9,10,11,12,13,14,month=1,year=1950 temp_3 temp_4

cdo    delete: Processed 12855 values from 1 variable over 12869 timesteps [1.62s 67MB].


In [7]:
# Make sure that the time is sorted correctly (sorttimestamp) and the file is named correctly.
! cdo sorttimestamp temp_4 ./Data_in_Netcdf_Format/eobsv23e_tg_3E_20E_45N_60N_1950_2020_only_Nov_Apr_lead_time_14d.nc

cdo    sorttimestamp (Warning): Time bounds unsupported by this operator, removed!
cdo    sorttimestamp: Processed 12855 values from 1 variable over 12855 timesteps [1.16s 59MB].


#### Convert from grib-format to netcdf-format

In [8]:
# Convert the grib-file to a netcdf-file if necessary. The Python-scripts are designed to use
# netcdf-files.
#! cdo -f nc copy ofile.grib ofile.nc

#### Remove unnecessary files

In [9]:
# Remove unnecessary files which have been created by CDO since the names of the input files 
# and output files have to be unique.
! rm temp*

## Continue with Python


For a nice overview of the data, pandas dataframes are used. These are then converted directly into csv-format for storage which ensures a safe and easy data transfer between various jupyter notebooks.

#### Define the paths' and files' names 

In [10]:
# Set the needed path and file names.
PATH_defined_functions = './Defined_Functions'

PATH_data = './Data_in_Netcdf_Format/'
ifile_data = 'eobsv23e_tg_3E_20E_45N_60N_1950_2020_only_Nov_Apr_lead_time_14d.nc'

PATH_thresholds = './Thresholds_Cold_Waves/'
ifile_thresholds = 'cold_wave_thresholds_Smid_et_al_2019_for_1970_2000.csv'

PATH_output_file = './Data_in_csv_Format/'
file_name_output_file = 'eobsv23e_tg_3E_20E_45N_60N_1950_2020_binary_cold_waves_lead_time_14d.csv'

#### Import the necessary packages and functions
Nothing needs to be changed here.

In [11]:
# Import the necessary python packages.
import numpy as np
import pandas as pd
import calendar
from datetime import datetime, timedelta

In [12]:
# Import the necessary defined functions.
import sys
sys.path.insert(1,PATH_defined_functions)
from read_in_netcdf_data import *
from read_in_csv_data import *
from create_auxiliary_date import *
from truncate_data_by_date import *
from apply_cold_wave_definition_smid_et_al_2019 import *

#### Read in the ground truth temperature data and check the file's content
Nothing needs to be changed here.

In [13]:
# Read in the ground truth and remove any unnamed columns as well as the index column.
df_data = read_in_netcdf_data(PATH_data,ifile_data)
df_data = df_data.loc[:, ~df_data.columns.str.contains('^Unnamed')]

In [14]:
# Show the head of the dataframe.
df_data.head()

,lat,lon,time,tg
0,0.0,0.0,1950-01-15,2.82
1,0.0,0.0,1950-01-16,2.31
2,0.0,0.0,1950-01-17,-0.69
3,0.0,0.0,1950-01-18,-3.72
4,0.0,0.0,1950-01-19,-6.43


In [15]:
# Show the end of the dataframe.
df_data.tail()

,lat,lon,time,tg
12850,0.0,0.0,2020-12-27,0.29
12851,0.0,0.0,2020-12-28,1.91
12852,0.0,0.0,2020-12-29,2.49
12853,0.0,0.0,2020-12-30,1.81
12854,0.0,0.0,2020-12-31,0.83


#### Create a minimal, useful representation of the data

In [16]:
# Remove any unnecessary columns here, e.g. the latitude and longitude for aerial means.
df_data = df_data.drop(['lat', 'lon'], axis =1 )

In [17]:
# For a usual data representation, convert e.g. the temperature from °Celsius to Kelvin. Note:
# Although the data is displayed (df_input_data.head()) with extra precision, it is saved
# correctly in csv-format later. 
data = np.array(df_data['tg']) 
data = data + 273.0
df_data['tg'] = data

#### Doublecheck the representation of the data

In [18]:
# Check if everything is sorted, renamed or removed correctly. Note:
# Although the data is displayed with wrong extra precision, it is saved correctly in
# csv-format later. 
df_data.head()

,time,tg
0,1950-01-15,275.820007
1,1950-01-16,275.309998
2,1950-01-17,272.309998
3,1950-01-18,269.279999
4,1950-01-19,266.570007


In [19]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_data.tail()

,time,tg
12850,2020-12-27,273.290009
12851,2020-12-28,274.910004
12852,2020-12-29,275.489990
12853,2020-12-30,274.809998
12854,2020-12-31,273.829987


In [20]:
# Set the name of the columns containing the time and the variables of the ground truth.
time_column_name = df_data.columns[0]
var_column_name = df_data.columns[1]

#### Read in the temperature thresholds of the cold wave criterion and check the file's content


In [21]:
# Read in the thresholds for the cold wave index and remove any unnamed columns as well as the
# index column.
df_thresholds = read_in_csv_data(PATH_thresholds,ifile_thresholds)
df_thresholds = df_thresholds.loc[:, ~df_thresholds.columns.str.contains('^Unnamed')]
df_thresholds = df_thresholds.drop(['index'], axis =1 )

In [22]:
# Show the head of the dataframe containing the cold wave thresholds.
df_thresholds.head()

,auxiliary_date,threshold_cold_wave
0,2003-11-01,277.684000
1,2003-11-02,277.465097
2,2003-11-03,277.391806
3,2003-11-04,277.402645
4,2003-11-05,277.355613


In [23]:
# Show the tail of the dataframe containing the thresholds.
df_thresholds.tail()

,auxiliary_date,threshold_cold_wave
177,2004-04-26,278.489871
178,2004-04-27,278.248129
179,2004-04-28,278.196000
180,2004-04-29,278.113613
181,2004-04-30,277.917613


In [24]:
# Set the name of the columns containing the time and the threshold.
time_column_name_thresholds = df_thresholds.columns[0]
var_column_name_thresholds = df_thresholds.columns[1]

#### Apply the cold wave criterion to the data
Nothing needs to be changed here.

In [25]:
# At first, two different dataframes are created with the threshold for the cold wave 
# definition. One for regular years and one for leap years. Therefore, the index of the original
# dataframe is set to the time and the index of the 29 February is determined. Then, a new 
# dataframe without the 29 February is created for regular years. The original dataframe is used
# for leap years.
df_thresholds[time_column_name_thresholds]=pd.to_datetime(df_thresholds[time_column_name_thresholds])
df_thresholds = df_thresholds.set_index(time_column_name_thresholds)
index_of_february_29 = df_thresholds[((df_thresholds.index.month == 2) & (df_thresholds.index.day == 29))].index
df_thresholds_without_29_feb = df_thresholds.drop(index_of_february_29)
df_thresholds = df_thresholds.reset_index()
df_thresholds_without_29_feb = df_thresholds_without_29_feb.reset_index()

In [26]:
# A list with all the start years of the winters in the training period is created.
start_years_of_winter = np.arange(1950, 2020)

In [27]:
# In case the ground truth timeseries does not start with the beginning of a winter, the beginning
#  of the timeseries until the start of the first winter is checked for cold waves. Therefore, the 
# respective part of the ground truth and the fitting part of the cold wave thresholds are extracted.
# Then, the cold wave definition by Smid et al. (2019) is applied and the binary classification
# of whether at a specific date a cold wave occurred ('1') or not ('0') is saved to a list. 
# Also, the dates are saved to a list.
dates = []

all_winters_list_cold_waves_data = []

start_winter = datetime(1950, 1, 15)
end_winter = datetime(1950, 4, 30)

df_thresholds_start_training_data = truncate_data_by_date(df_thresholds, time_column_name_thresholds, start_winter.strftime('%Y_%m_%d'), end_winter.strftime('%Y_%m_%d')) 
threshold_cold_waves = df_thresholds_start_training_data[var_column_name_thresholds]


df_data_respective_winter = truncate_data_by_date(df_data, time_column_name, start_winter.strftime('%Y_%m_%d'), end_winter.strftime('%Y_%m_%d')) 

df_data_binned = pd.DataFrame()
df_data_binned['time'] = df_data_respective_winter[time_column_name]
list_cold_waves_data = apply_cold_wave_definition_smid_et_al_2019(df_data_binned, df_data_respective_winter, var_column_name, threshold_cold_waves)
          
all_winters_list_cold_waves_data.extend(list_cold_waves_data)
dates.extend(pd.to_datetime(df_data_respective_winter[time_column_name]))

In [28]:
# Now, the same is done for every complete winter in the ground truth timeseries. The cold wave
# thresholds are taken depending on whether it is a leap year or not, meaning that the 
# 29 February is included in the threshold data or not.
for start_year_of_winter in start_years_of_winter:
    
    if calendar.isleap(start_year_of_winter+1):
        threshold_cold_waves = df_thresholds[var_column_name_thresholds]
    else:
        threshold_cold_waves = df_thresholds_without_29_feb[var_column_name_thresholds]
    
    start_winter = datetime(start_year_of_winter, 11, 1)
    end_winter = datetime(start_year_of_winter+1, 4, 30)

    df_data_respective_winter = truncate_data_by_date(df_data, time_column_name, start_winter.strftime('%Y_%m_%d'), end_winter.strftime('%Y_%m_%d')) 

    df_data_binned = pd.DataFrame()
    df_data_binned['time'] = df_data_respective_winter[time_column_name]
    list_cold_waves_data = apply_cold_wave_definition_smid_et_al_2019(df_data_binned, df_data_respective_winter, var_column_name, threshold_cold_waves)
          
    all_winters_list_cold_waves_data.extend(list_cold_waves_data)

    dates.extend(pd.to_datetime(df_data_respective_winter[time_column_name]))

In [29]:
# In a last step, the procedure is done for the rest of the ground truth timeseries which does
# not cover a whole winter anymore.
start_winter = datetime(2020, 11, 1)
end_winter = datetime(2020, 12, 31)

df_thresholds_end_training_data = truncate_data_by_date(df_thresholds, time_column_name_thresholds, start_winter.strftime('%Y_%m_%d'), end_winter.strftime('%Y_%m_%d')) 
threshold_cold_waves = df_thresholds_end_training_data[var_column_name_thresholds]


df_data_respective_winter = truncate_data_by_date(df_data, time_column_name, start_winter.strftime('%Y_%m_%d'), end_winter.strftime('%Y_%m_%d')) 

df_data_binned = pd.DataFrame()
df_data_binned['time'] = df_data_respective_winter[time_column_name]
list_cold_waves_data = apply_cold_wave_definition_smid_et_al_2019(df_data_binned, df_data_respective_winter, var_column_name, threshold_cold_waves)
          
all_winters_list_cold_waves_data.extend(list_cold_waves_data)

dates.extend(pd.to_datetime(df_data_respective_winter[time_column_name]))

In [30]:
# Now, a new dataframe is created containing the dates of the ground truth and the binary cold
# wave index.
df_data_binary_cold_waves = pd.DataFrame()
df_data_binary_cold_waves['time'] = dates
df_data_binary_cold_waves['Cold_Wave'] = all_winters_list_cold_waves_data

#### Plausibility check if the application of the cold wave criterion worked
Nothing needs to be changed here.

In [31]:
# Here it is checked whether all dates in the ground truth timeseries have been checked for a
# cold wave occurrence.
if len(df_data_binary_cold_waves['Cold_Wave']) == len(df_data[var_column_name]):
    print('All data has been checked for cold waves.')
else:
    print('Not all data has been checked for cold waves!')
    print('Days to be checked: '+str(len(df_data[var_column_name])))
    print('Days actually checked: '+str(len(all_winters_list_cold_waves_data)))

All data has been checked for cold waves.


In [32]:
# To check if the cold wave definition has been correctly, it is checked whether a reasonable
# number of days with cold waves have been detected in the ground truth timeseries.
days_with_cold_waves = 0
binary_list_of_cold_waves = df_data_binary_cold_waves['Cold_Wave']
for i in range(len(binary_list_of_cold_waves)):
    if binary_list_of_cold_waves[i] == 1:
        days_with_cold_waves +=1

print('There are '+str(days_with_cold_waves)+' days with cold waves from a total of '+str(len(binary_list_of_cold_waves))+' days.')
print('This means that '+str(days_with_cold_waves/len(binary_list_of_cold_waves)*100)+'% of the winterdays are cold wave days.')

There are 2697 days with cold waves from a total of 12855 days.
This means that 20.980163360560095% of the winterdays are cold wave days.


#### Doublecheck the representation of the data

In [33]:
# Check if everything is sorted, renamed or removed correctly.
df_data_binary_cold_waves.head()

,time,Cold_Wave
0,1950-01-15,0
1,1950-01-16,0
2,1950-01-17,0
3,1950-01-18,0
4,1950-01-19,0


In [34]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_data_binary_cold_waves.tail()

,time,Cold_Wave
12850,2020-12-27,0
12851,2020-12-28,0
12852,2020-12-29,0
12853,2020-12-30,0
12854,2020-12-31,0


#### Save the binary ground truth cold wave data
Nothing needs to be changed here.

In [35]:
# Save the pandas dataframe in csv-format.
df_data_binary_cold_waves.to_csv(PATH_output_file+file_name_output_file)

In [36]:
# End of Program